In [8]:
import numpy as np
import pandas as pd
import cv2
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import torch
from torch.utils.data import Dataset, DataLoader

In [9]:
# === Load CSV ===
csv_path = "archive/english.csv"
df = pd.read_csv(csv_path)

# === Encode labels ===
label_encoder = LabelEncoder()
df['label'] = label_encoder.fit_transform(df['label'])
num_classes = len(label_encoder.classes_)


In [10]:
# === Custom Dataset Class ===
class CharDataset(Dataset):
    def __init__(self, df, img_dir):
        self.df = df
        self.img_dir = img_dir
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        img_path = os.path.join("archive", self.df.iloc[idx, 0])  # path like archive/Img/img001-001.png
        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        img = cv2.resize(img, (28, 28))            # resize to 28x28
        img = img.astype(np.float32) / 255.0       # normalize 0–1
        img = img.flatten()                        # flatten to vector (784,)
        label = self.df.iloc[idx, 1]
        return torch.tensor(img), torch.tensor(label)

In [11]:
# === Train / Test Split ===
train_df, test_df = train_test_split(df, test_size=0.2, stratify=df['label'], random_state=42)

train_dataset = CharDataset(train_df, "archive")
test_dataset = CharDataset(test_df, "archive")

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

print("Dataset ready ✅")
print("Train samples:", len(train_dataset), " Test samples:", len(test_dataset))

Dataset ready ✅
Train samples: 2728  Test samples: 682


In [15]:
import torch
import torch.nn as nn
import torch.nn.functional as F

num_classes = df['label'].nunique()   # total unique characters

class MLP(nn.Module):
    def __init__(self, input_size=784, hidden1=128, hidden2=64, num_classes=num_classes):
        super(MLP, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden1)
        self.fc2 = nn.Linear(hidden1, hidden2)
        self.fc3 = nn.Linear(hidden2, num_classes)
    
    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)   # logits (raw scores)
        return x

# Instantiate model
model = MLP()
print(model)


MLP(
  (fc1): Linear(in_features=784, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=64, bias=True)
  (fc3): Linear(in_features=64, out_features=62, bias=True)
)


In [17]:
import torch.optim as optim

# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# Move model to device
model = model.to(device)

# Loss function & optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)  # you can tune lr later


Using device: cpu


In [18]:
def train_model(model, train_loader, test_loader, criterion, optimizer, epochs=10):
    for epoch in range(1, epochs+1):
        model.train()
        train_loss, correct, total = 0, 0, 0
        
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            
            # Forward pass
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            
            # Backward + optimize
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
            
            # Accuracy
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
        
        train_acc = 100 * correct / total
        
        # Validation accuracy
        val_acc = evaluate(model, test_loader)
        
        print(f"[Epoch {epoch:2d}] "
              f"Train Loss={train_loss/len(train_loader):.4f} "
              f"Train Acc={train_acc:.2f}% "
              f"Val Acc={val_acc:.2f}%")

# Helper function for evaluation
def evaluate(model, loader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = model(imgs)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    return 100 * correct / total


In [19]:
train_model(model, train_loader, test_loader, criterion, optimizer, epochs=10)

[Epoch  1] Train Loss=4.1378 Train Acc=1.58% Val Acc=1.61%
[Epoch  2] Train Loss=4.1302 Train Acc=1.69% Val Acc=1.61%
[Epoch  3] Train Loss=4.1267 Train Acc=1.58% Val Acc=1.91%
[Epoch  4] Train Loss=4.1198 Train Acc=1.87% Val Acc=1.76%
[Epoch  5] Train Loss=4.0956 Train Acc=3.30% Val Acc=4.11%
[Epoch  6] Train Loss=4.0274 Train Acc=3.81% Val Acc=5.87%
[Epoch  7] Train Loss=3.9179 Train Acc=5.39% Val Acc=5.57%
[Epoch  8] Train Loss=3.8024 Train Acc=5.50% Val Acc=5.87%
[Epoch  9] Train Loss=3.7343 Train Acc=6.85% Val Acc=7.48%
[Epoch 10] Train Loss=3.6799 Train Acc=7.51% Val Acc=7.04%


In [20]:
param_grid = [
    {"hidden_size": 128, "lr": 0.01, "batch_size": 64},
    {"hidden_size": 256, "lr": 0.001, "batch_size": 64},
    {"hidden_size": 512, "lr": 0.001, "batch_size": 128},
]

best_acc = 0
best_params = None

for params in param_grid:
    print("\n=== Testing:", params)
    
    # Update model with params
    model = MLP(input_size, params["hidden_size"], output_size).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=params["lr"])
    criterion = nn.CrossEntropyLoss()
    
    # Update loader with batch size
    train_loader = DataLoader(train_dataset, batch_size=params["batch_size"], shuffle=True)
    
    # Train
    history = train_model(model, train_loader, test_loader, criterion, optimizer, epochs=5)
    
    # Evaluate on test set
    acc = evaluate(model, test_loader)
    print(f"Test Accuracy: {acc:.2f}%")
    
    if acc > best_acc:
        best_acc = acc
        best_params = params

print("\n✅ Best Params:", best_params, " with Accuracy:", best_acc)



=== Testing: {'hidden_size': 128, 'lr': 0.01, 'batch_size': 64}


NameError: name 'input_size' is not defined